# 分板块研究平均偏度、峰度影响：中小板

In [1]:
## 导入NumPy、Pandas、Wind
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime
from dateutil.relativedelta import relativedelta

In [2]:
## 读入个股收益率各月的方差、偏度、峰度
prdct = pd.read_csv('Raw Transaction Data/prdct_dt.csv')
## 读入上市公司代码与上市板块
board = pd.read_csv('Raw Transaction Data/AllCorps.csv')
## 合并指标值与板块信息
prdct_mk = pd.merge(prdct, board, left_on = 'code', right_on = 'codes', how = 'left')

In [3]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [4]:
## 读取shibor数据
shibor = w.wsd("SHIBORON.IR", "open", "2007-01-01", "2019-01-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor['shibor'] = shibor['shibor'] * 0.01
shibor['date'] = pd.to_datetime(shibor['date'])
shibor['year'] = shibor['date'].dt.year
shibor['month'] = shibor['date'].dt.month

In [5]:
## 计算各月shibor均值
intrst = shibor.groupby([shibor['year'], shibor['month']])['shibor'].agg([('mn_SHIBOR', 'mean'), ])
intrst = intrst.reset_index()
intrst = intrst.reset_index()
intrst.columns = ['index', 'year', 'month', 'shibor']

In [6]:
w.wsd("399101.SZ,399102.SZ", "swing", "2021-03-13", "2021-04-11", "TradingCalendar=SZSE")

.ErrorCode=0
.Codes=[399101.SZ,399102.SZ]
.Fields=[SWING]
.Times=[20210315,20210316,20210317,20210318,20210319,20210322,20210323,20210324,20210325,20210326,...]
.Data=[[2.0357938766925394,1.4297563128116058,2.070395878617105,0.9953227892641926,1.8636297475490646,1.5031558514822783,2.003200654375499,1.9174195374576768,1.5205822866697245,2.1270038458947242,...],[3.1995992585594935,1.6053292275110314,2.4202364816934505,1.1775626270564108,1.8737409938611518,1.557455746612023,1.9987918433135274,1.7575514099637886,2.1908016006818825,2.520587102028522,...]]

In [7]:
## 保留中小板指标
prdct_2 = prdct_mk[prdct_mk['boardtype'] == '中小板']
prdct_2 = prdct_2.reset_index(drop = True)

In [8]:
codes = prdct_2['code'].tolist()
codes = list(set(codes))
len(codes)

922

## 计算领先一期的超额收益率

In [9]:
## 指数月收益数据
mkt2_r = w.wsd("399101.SZ", "pct_chg", "2007-01-01", "2019-01-31", "Period=M;TradingCalendar=SZSE")
mkt2_r = pd.DataFrame(mkt2_r.Data, index = mkt2_r.Codes, columns = mkt2_r.Times)
mkt2_r = mkt2_r.T
mkt2_r = mkt2_r.reset_index()
mkt2_r = mkt2_r.reset_index()
mkt2_r.columns = ['index', 'date', 'pct_chg']
mkt2_r['pct_chg'] = mkt2_r['pct_chg'] * 0.01

In [28]:
## 计算月超额收益率
mkt2_exr = pd.merge(mkt2_r, intrst, on = 'index', how = 'left')
mkt2_exr['exr'] = mkt2_exr['pct_chg'] - mkt2_exr['shibor']
mkt2_exr = mkt2_exr[['index', 'exr', 'date']]

In [29]:
mkt2_r0 = mkt2_exr.head(144)
mkt2_r0.columns = ['index', 'r0', 'date']

In [30]:
## 删除首行，得到领先一期的收益率
mkt2_exr = mkt2_exr[['exr']]
mkt2_exr = mkt2_exr.drop(0, axis = 0)
mkt2_exr = mkt2_exr.reset_index(drop = True)
mkt2_exr = mkt2_exr.reset_index()

## 计算市场方差、偏度、峰度

In [13]:
## 获得中小板综指的指数数据
trdt_mk2 = w.wsd("399101.SZ", "swing", "2007-01-01", "2019-01-31", "TradingCalendar=SZSE")
trdt_mk2 = pd.DataFrame(trdt_mk2.Data, index = trdt_mk2.Codes, columns = trdt_mk2.Times)
trdt_mk2 = trdt_mk2.T
trdt_mk2 = trdt_mk2.reset_index()
trdt_mk2.columns = ['date', 'pct_chg']
trdt_mk2['pct_chg'] = trdt_mk2['pct_chg'] * 0.01
trdt_mk2['date'] = pd.to_datetime(trdt_mk2['date'])
trdt_mk2['year'] = trdt_mk2['date'].dt.year
trdt_mk2['month'] = trdt_mk2['date'].dt.month

In [14]:
## 合并中小板综指数据和shibor收益数据
dt_mk2 = pd.merge(trdt_mk2, shibor[['date', 'shibor']], on = 'date', how = 'left')
dt_mk2['exr'] = dt_mk2['pct_chg'] - dt_mk2['shibor']
dt_mk2 = dt_mk2[['year', 'month', 'date', 'pct_chg', 'shibor', 'exr']]

In [15]:
## 分组，计算各月均值、方差、偏度、峰度
sta_mk2 = dt_mk2.groupby([dt_mk2['year'], dt_mk2['month']])['exr'].agg([('num', 'count'), ('mean', 'mean'), ('math_var', 'var'), ('math_skew', 'skew'),('math_kurt', stats.kurtosis),])
sta_mk2 = sta_mk2.reset_index()

In [16]:
## 删除首行
mkt_lf2 = dt_mk2.groupby([dt_mk2['year'], dt_mk2['month']])['date'].first()
mkt_lf2 = mkt_lf2.reset_index()
mkt_lf2.insert(2, 'flag', np.ones(len(mkt_lf2)))
mkt_lf2 = mkt_lf2[['date', 'flag']]

## 保留其他数据
mkt_wof2 = pd.merge(dt_mk2, mkt_lf2, on = 'date', how = 'left')
mkt_wof2 = mkt_wof2[mkt_wof2['flag'] != 1]
mkt_wof2 = mkt_wof2[['date', 'exr']]
mkt_wof2 = mkt_wof2.rename(columns={'exr': 'exr_f'})
mkt_wof2.insert(2, 'order', np.arange(len(mkt_wof2)))

## 删除末行
mkt_ll2 = dt_mk2.groupby([dt_mk2['year'], dt_mk2['month']])['date'].last()
mkt_ll2 = mkt_ll2.reset_index()
mkt_ll2.insert(2, 'flag', np.ones(len(mkt_ll2)))
mkt_ll2 = mkt_ll2[['date', 'flag']]

## 保留其他数据
mkt_wol2 = pd.merge(dt_mk2, mkt_ll2, on = 'date', how = 'left')
mkt_wol2 = mkt_wol2[mkt_wol2['flag'] != 1]
mkt_wol2 = mkt_wol2[['year', 'month', 'exr']]
mkt_wol2 = mkt_wol2.rename(columns={'exr': 'exr_l'})
mkt_wol2.insert(2, 'order', np.arange(len(mkt_wol2)))

## 合并去除第一行/最后一行的数据，准备计算调整的方差
## 注意，保留的日期系exr_f的日期
ajsta0_mk2 = pd.merge(mkt_wol2, mkt_wof2, on = 'order', how = 'left')
ajsta0_mk2 = ajsta0_mk2[[ 'year', 'month', 'date', 'exr_f', 'exr_l']]

## 合并均值数据
ajsta_mk2 = pd.merge(ajsta0_mk2, sta_mk2[['year', 'month', 'mean']], on = ['year', 'month'], how = 'left')
ajsta_mk2['diff1'] = ajsta_mk2['exr_f'] - ajsta_mk2['mean']
ajsta_mk2['diff2'] = ajsta_mk2['exr_l'] - ajsta_mk2['mean']
ajsta_mk2['product'] = ajsta_mk2['diff1'] * ajsta_mk2['diff2']
ajsta_mk2 = ajsta_mk2.groupby([ajsta_mk2['year'], ajsta_mk2['month']])['product'].agg([('prdct', 'sum'),])
ajsta_mk2 = ajsta_mk2.reset_index()

## 合并调整后的数据和原统计指标值
ajs_mk2 = pd.merge(sta_mk2, ajsta_mk2, on = ['year', 'month'], how = 'left' )
ajs_mk2['var'] = ajs_mk2['math_var'] * ajs_mk2['num'] + 2* ajs_mk2['prdct']
ajs_mk2['adj_var'] = ajs_mk2['var']/ajs_mk2['num']

## 保留数据
prdct_mk2 = ajs_mk2[['year', 'month', 'num', 'mean', 'var', 'adj_var', 'math_skew', 'math_kurt']]
prdct_mk2 = prdct_mk2.reset_index()
prdct_mk2 = prdct_mk2[['index', 'var', 'math_skew', 'math_kurt']]
prdct_mk2.columns = ['index', 'mkt_var', 'mkt_skew', 'mkt_kurt']

## 等权重加权法

In [17]:
## 计算中小板平均方差、偏度与峰度
mnv_ew2 = prdct_2.groupby([prdct['year'], prdct['month']])['var', 'adj_var', 'math_skew', 'math_kurt'].mean()
mnv_ew2 = mnv_ew2.reset_index()
mnv_ew2 = mnv_ew2.reset_index()
mnv_ew2 = mnv_ew2[['index', 'adj_var', 'math_skew', 'math_kurt']]
mnv_ew2.columns = ['index', 'var_ew', 'skew_ew', 'kurt_ew']

In [18]:
mnv_ew2

,index,var_ew,skew_ew,kurt_ew
0,0,0.000980,-0.043556,0.298445
1,1,0.002265,-0.043980,0.594932
2,2,0.003053,0.002185,0.436344
3,3,0.001991,-0.035342,0.545561
4,4,0.001212,-0.041773,0.546637
...,...,...,...,...
139,139,0.002604,0.021512,0.443811
140,140,0.001029,0.012965,0.462252
141,141,0.001229,-0.010754,0.450289
142,142,0.000953,-0.025806,0.459574


## 市值加权法

In [19]:
## 读取市值数据
evdt = w.wsd(codes, "ev", "2006-12-01", "2018-11-30", "unit=1;Period=M;TradingCalendar=SZSE")
ev = pd.DataFrame(evdt.Data, index = evdt.Codes, columns = evdt.Times)
ev = ev.T
ev = ev.reset_index()

In [20]:
## 转为一维表，准备合并
ev_T = pd.melt(ev, 
            id_vars = 'index', 
            value_vars = list(ev.columns[1:]), 
            var_name='code', 
            value_name='ev_month')
ev_T.columns = ['date', 'code', 'ev_month']
ev_T['date'] = pd.to_datetime(ev_T['date'])
ev_T['date'] = ev_T['date'].apply(lambda x: x + relativedelta(months = +1)) 
## 生成年、月
ev_T['year'] = ev_T['date'].dt.year
ev_T['month'] = ev_T['date'].dt.month

In [21]:
## 合并prdct和ev_w
vw_fore = pd.merge(prdct_2, ev_T, on = ['code', 'year', 'month'], how = 'left')
vw_fore = vw_fore.dropna().reset_index(drop = True)

## 按月分组，计算各月总市值
## 计算样本数据的权重
tt_ev = vw_fore.groupby([vw_fore['year'], vw_fore['month']])['ev_month'].agg([('TEV', 'sum'),])
tt_ev = tt_ev.reset_index()
ev_w = pd.merge(vw_fore, tt_ev, on = ['year', 'month'], how = 'left')
ev_w['weight'] = ev_w['ev_month']/ ev_w['TEV']
ev_w = ev_w.sort_values(by= ["code", 'year', 'month'] , ascending = True)

In [22]:
ev_w

,code,year,month,num,var,adj_var,math_skew,math_kurt,codes,boardtype,date,ev_month,TEV,weight
0,002001.SZ,2007,1,20,0.044163,0.002208,-0.128903,-0.260546,002001.SZ,中小板,2007-01-29,1.727403e+09,2.015296e+11,0.008571
1,002001.SZ,2007,2,15,0.007761,0.000517,-1.132279,1.666986,002001.SZ,中小板,2007-02-28,1.994210e+09,2.773745e+11,0.007190
2,002001.SZ,2007,3,22,0.014954,0.000680,-0.186330,-1.096533,002001.SZ,中小板,2007-03-28,2.329429e+09,2.961480e+11,0.007866
3,002001.SZ,2007,4,21,0.017267,0.000822,-0.512604,2.105186,002001.SZ,中小板,2007-04-30,2.924613e+09,3.524963e+11,0.008297
4,002001.SZ,2007,5,18,0.040734,0.002263,-0.566723,-0.546197,002001.SZ,中小板,2007-05-30,3.208523e+09,4.572322e+11,0.007017
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85019,002939.SZ,2018,12,20,0.004240,0.000212,-0.088262,-0.599453,002939.SZ,中小板,2018-12-30,3.246162e+10,7.476731e+12,0.004342
85020,002940.SZ,2018,11,22,0.014260,0.000648,0.892160,1.190855,002940.SZ,中小板,2018-11-30,4.590000e+09,7.247250e+12,0.000633
85021,002940.SZ,2018,12,20,0.008783,0.000439,0.378156,-0.894386,002940.SZ,中小板,2018-12-30,3.627900e+09,7.476731e+12,0.000485
85022,002941.SZ,2018,12,20,0.115405,0.005770,-0.171631,-1.449251,002941.SZ,中小板,2018-12-30,8.068950e+09,7.476731e+12,0.001079


In [23]:
## 筛选需要的指标列
vw = ev_w[['code', 'year', 'month', 'weight', 'var', 'adj_var', 'math_skew', 'math_kurt']]

## 计算加权指标
vw['var_vw'] = vw['adj_var'] * vw['weight']
vw['skew_vw'] = vw['math_skew'] * vw['weight']
vw['kurt_vw'] = vw['math_kurt'] * vw['weight']

## 按月加总
mnv_vw = vw.groupby([vw['year'], vw['month']])['var_vw', 'skew_vw', 'kurt_vw'].sum()
mnv_vw = mnv_vw.reset_index()
mnv_vw = mnv_vw.reset_index()
mnv_vw = mnv_vw[['index', 'var_vw', 'skew_vw', 'kurt_vw']]

In [33]:
mk2_reg = pd.merge(mkt2_exr, mkt2_r0, on = 'index', how = 'left')
mk2_reg = pd.merge(mk2_reg, prdct_mk2, on = 'index', how = 'left')
mk2_reg = pd.merge(mk2_reg, mnv_ew2, on = 'index', how = 'left')
mk2_reg = pd.merge(mk2_reg, mnv_vw, on = 'index', how = 'left')
mk2_reg = mk2_reg.drop('index', axis = 1)
mk2_reg.columns = ['r', 'r0', 'date', 'mkt_var', 'mkt_skew', 'mkt_kurt', 'var_ew', 'skew_ew', 'kurt_ew', 'var_vw', 'skew_vw', 'kurt_vw']

In [36]:
mk2_reg.head(96).to_csv("Plot/reg_mkt2.csv", index = False) 

In [35]:
mk2_reg.head(96)

,r,r0,date,mkt_var,mkt_skew,mkt_kurt,var_ew,skew_ew,kurt_ew,var_vw,skew_vw,kurt_vw
0,0.009802,0.230003,2007-01-31,0.005553,0.547110,-0.503697,0.000980,-0.043556,0.298445,0.001738,-0.086166,-0.469109
1,0.107716,0.009802,2007-02-28,0.013603,2.175925,4.111329,0.002265,-0.043980,0.594932,0.000749,-0.908798,1.604237
2,0.157488,0.107716,2007-03-30,0.002252,0.433565,-0.806941,0.003053,0.002185,0.436344,0.000805,0.291905,0.296489
3,0.071053,0.157488,2007-04-30,0.004664,1.220859,2.583147,0.001991,-0.035342,0.545561,0.001225,-0.039060,0.523121
4,-0.082590,0.071053,2007-05-31,0.008649,1.516893,1.061753,0.001212,-0.041773,0.546637,0.001337,-0.462958,0.697041
...,...,...,...,...,...,...,...,...,...,...,...,...
91,0.082114,0.023426,2014-08-29,0.000126,1.975884,4.536456,0.001000,-0.095961,0.449546,0.000430,0.370991,0.552244
92,-0.018651,0.082114,2014-09-30,0.000889,3.049358,8.527519,0.000917,-0.038538,0.366160,0.000539,0.073669,1.387098
93,-0.002421,-0.018651,2014-10-31,0.000520,1.519136,2.151719,0.001030,-0.090721,0.561121,0.000581,0.335618,0.147435
94,-0.084142,-0.002421,2014-11-28,0.001583,2.383356,5.044192,0.000984,-0.062438,0.752351,0.000453,0.076322,0.732126
